# Inference

In [2]:
%uv pip install torchcodec==0.7
%uv pip install datasets==4.4.1 
%uv pip install numpy==2.1.2
%uv pip install pandas==2.3.2
%uv pip install speechbrain

Using Python 3.12.6 environment at: /usr/local
Resolved 1 package in 32ms
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
Prepared 1 package in 58ms
Installed 1 package in 91ms
 + torchcodec==0.7.0
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.6 environment at: /usr/local
Resolved 36 packages in 184ms
⠙ Preparing packages... (0/5)
⠙ Preparing packages... (0/5)
⠙ Preparing packages... (0/5)
⠙ Preparing packages... (0/5)
⠙ Preparing packages... (0/5)
dill       ------

In [1]:
import torch
from speechbrain.pretrained import EncoderASR
from datasets import load_dataset

/usr/local/lib/python3.12/site-packages/speechbrain/utils/torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()
/tmp/ipykernel_372/3657245855.py:2: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'speechbrain.inference'. Please update your script. This is a change from SpeechBrain 1.0. See: https://github.com/speechbrain/speechbrain/releases/tag/v1.0.0
  from speechbrain.pretrained import EncoderDecoderASR


In [2]:
# Load dataset
ds = load_dataset("MohamedGomaa30/Egyptian-ASR-MGB-3", split="test")

README.md:   0%|          | 0.00/584 [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/297M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/304M [00:00<?, ?B/s]

data/eval-00000-of-00001.parquet:   0%|          | 0.00/161M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/159M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/759 [00:00<?, ? examples/s]

Generating eval split:   0%|          | 0/200 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/200 [00:00<?, ? examples/s]

In [3]:
ds

Dataset({
    features: ['audio', 'sentence'],
    num_rows: 200
})

In [4]:
ds[0]

{'audio': <datasets.features._torchcodec.AudioDecoder at 0x2a7c8933d1c0>,
 'sentence': 'وأين هي طرق التنمية وأين هي طرق التدريس التي تحفظ هذا الحق حق التعليم الفني يحتاج إلى كرامة أولاً ثم إلى طرق تدريس ثانياً وقبل هذا كله وبعده يحتاج الى المخلصون من هذا الوطن الذين يريدون حقاً لهذه القاطرة أن تنطلق'}

In [5]:
ds[0]['audio']['array']

array([-8.1787109e-03, -8.7280273e-03, -9.2468262e-03, ...,
        6.1035156e-05,  2.4414062e-04, -1.8310547e-04], dtype=float32)

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [13]:

# 1. Load the SpeechBrain model (CTC-based model)
MODEL = "speechbrain/asr-wav2vec2-commonvoice-14-ar"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading model: {MODEL}")
print(f"Using device: {device}")

# Load model with run_opts to specify device
# Use EncoderASR instead of EncoderDecoderASR for CTC models
asr_model = EncoderASR.from_hparams(
    source=MODEL,
    savedir=f"pretrained_models/{MODEL.split('/')[-1]}",
    run_opts={"device": device}
)

# 3. Inference Loop
print("\nStarting transcription...")
for i, sample in enumerate(ds):
    if i >= 10: 
        break
    
    # Get audio array and sample rate
    audio_array = sample["audio"]["array"]
    sample_rate = sample["audio"]["sampling_rate"]
    
    # Convert to torch tensor
    audio_tensor = torch.FloatTensor(audio_array)
    
    with torch.no_grad():
        try:
            # Transcribe (SpeechBrain handles single audio at a time)
            # Pass audio on same device as model
            # SpeechBrain expects relative lengths (0-1 range)
            audio_batch = audio_tensor.unsqueeze(0).to(device)  # Add batch dimension
            audio_len = torch.tensor([1.0]).to(device)  # Relative length (full audio = 1.0)
            
            predicted_words, predicted_tokens = asr_model.transcribe_batch(
                audio_batch,
                audio_len
            )
            
            # Extract transcription text
            result_text = predicted_words[0]
            
            # Get reference text
            reference = sample.get('transcription') or sample.get('text') or sample.get('sentence') or 'N/A'
            
            print(f"--- Sample {i+1} ---")
            print(f"Reference:     {reference}")
            print(f"Transcription: {result_text}")
            print("-" * 40)
            
        except Exception as e:
            print(f"Error on sample {i+1}: {e}")
            import traceback
            traceback.print_exc()
            continue

print("\nTranscription complete!")

Loading model: speechbrain/asr-wav2vec2-commonvoice-14-ar
Using device: cuda


speechbrain.lobes.models.huggingface_transformers.huggingface - Wav2Vec2Model is frozen.


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]


Starting transcription...
--- Sample 1 ---
Reference:     وأين هي طرق التنمية وأين هي طرق التدريس التي تحفظ هذا الحق حق التعليم الفني يحتاج إلى كرامة أولاً ثم إلى طرق تدريس ثانياً وقبل هذا كله وبعده يحتاج الى المخلصون من هذا الوطن الذين يريدون حقاً لهذه القاطرة أن تنطلق
Transcription: التعليم وإن هي الطرق التمية وإن هيطلق التدريس التي تحفظ هذا الحق حق التعليم الفن يحتاب إلى كرامة أولا ثم إلى تلقق تدريس ثانين وقبل هذا كله وبعدة يحتاب إلى المخلصون من هذا الوطن الذين يريدون حقا لهذه القاترة أن تنطلق
----------------------------------------
--- Sample 2 ---
Reference:     تمام و ادي ال اللون أنا ممكن اللون أخفف كده بس شوية صغيرين دي بقىأنا هخلي لونها أفتح شوية من اللي أنا عاملاها ومجهزاها معايا بنورك يا حبيبتي ... والله العظيم كلنا بنتفرج على حضرتك صح إحنا مقصرين في الاتصال
Transcription: وأيدي إلاون أنا ممكنون أخف الكذببة سشويس غيرية اجبها خليل منهاافتح شوايا من إلا لناملها وهداه معايةحد كريمريما ال الزايف أن المعليتالكم السلام فيك بات عن ذلك تأقرب لي وى واللهمننور الششكل و لنورك ولا بال